# Workshop — 5. Add an Agentic Supervisor

The fine-tuned SmolVLA remains the robot policy: it converts images, robot state, and language into joint actions. This notebook adds a Strands Agent **above** that policy.

```text
user goal
   ↓
Strands Agent (inspect, decide, continue or stop)
   ↓ tool call
strands-robots run_policy(...)
   ↓ 30 Hz control loop
SmolVLA policy → joint actions → MuJoCo
   ↑                         ↓
   └──── images + state ─────┘
```

All components of this loop are defined visibly below. The reusable implementation in `code/agentic_pick.py` remains available, but this notebook does not import it.

## 1. Install and Configure the Runtime

The local VLA uses the workshop environment. The high-level Strands Agent uses an Amazon Bedrock model by default, so AWS credentials and Bedrock access are also required.

In [ ]:
%pip install -q -r requirements.txt

### Configure local rendering and inference

These settings select the correct MuJoCo renderer, enable PyTorch fallbacks on Apple Silicon, and expose Homebrew FFmpeg to the notebook process. They must be set before importing the robotics runtime.

In [ ]:
import os
import platform

os.environ.setdefault("MUJOCO_GL", "cgl" if platform.system() == "Darwin" else "egl")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("STRANDS_TRUST_REMOTE_CODE", "1")
if platform.system() == "Darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")

## 2. Locate the Fine-Tuned Checkpoint

Notebook 4 downloaded the SageMaker artifact under `outputs/smolvla-so100/evaluations/<training-job>/model`. We select the most recently modified complete local checkpoint rather than downloading it again.

In [ ]:
from pathlib import Path

MODEL_SEARCH_ROOT = Path("outputs/smolvla-so100/evaluations").resolve()
model_candidates = sorted(
    (config_path.parent for config_path in MODEL_SEARCH_ROOT.glob("*/model/config.json")),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
if not model_candidates:
    raise FileNotFoundError(
        f"No downloaded fine-tuned checkpoint found under {MODEL_SEARCH_ROOT}. "
        "Run Notebook 4 first."
    )

LOCAL_MODEL_DIR = model_candidates[0]
print("Fine-tuned checkpoint:", LOCAL_MODEL_DIR)

## 3. Define and Build the Simulation

The agent needs the same experiment used for evaluation. We define the robot contract, reconstruct the target tray, add the two cameras, restore the dataset-aligned start pose, and implement the deterministic task metric.

In [ ]:
import mujoco
import numpy as np
import torch
from strands_robots import Robot
from strands_robots.policies import create_policy

INSTRUCTION = "Pick up the cube and place it in the box."
JOINT_KEYS = ["Rotation", "Pitch", "Elbow", "Wrist_Pitch", "Wrist_Roll", "Jaw"]
DATASET_MEAN_STATE = np.array(
    [14.4717, -55.7695, 54.3857, 63.2262, 85.8417, 9.3545],
    dtype=np.float64,
)
GRIPPER_JOINT_RANGE = (-0.175, 1.745)
BOX_CENTER = np.array([0.16, -0.30], dtype=np.float64)
FPS = 30


def require_success(result, operation):
    """Raise when a strands-robots operation returns an error envelope."""
    if result.get("status") == "success":
        return result
    text = " | ".join(
        str(item.get("text"))
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )
    raise RuntimeError(f"{operation} failed: {text or result}")


def dataset_mean_action():
    """Convert the original real-data mean pose to MuJoCo units."""
    values = np.empty(6, dtype=np.float64)
    values[:5] = np.deg2rad(DATASET_MEAN_STATE[:5])
    jaw_min, jaw_max = GRIPPER_JOINT_RANGE
    values[5] = jaw_min + (DATASET_MEAN_STATE[5] / 100.0) * (jaw_max - jaw_min)
    return dict(zip(JOINT_KEYS, values.tolist(), strict=True))

### Instantiate the physical experiment

The scene builder makes the experiment repeatable: it adds the same cube, tray, cameras, and start pose used during evaluation. The diagnostic function remains separate so task success can be checked without asking the LLM.

In [ ]:
def build_scene():
    """Create one fresh SO100 pick-place scene."""
    sim = Robot("so100", mesh=False)
    require_success(
        sim.add_object(
            name="cube",
            shape="box",
            position=[0.0, -0.35, 0.015],
            size=[0.03, 0.03, 0.03],
            color=[0.9, 0.12, 0.08, 1.0],
            mass=0.03,
        ),
        "add cube",
    )
    blue = [0.12, 0.28, 0.75, 1.0]
    x, y = BOX_CENTER
    parts = {
        "target_base": ([x, y, 0.005], [0.12, 0.12, 0.01]),
        "target_left": ([x - 0.055, y, 0.025], [0.01, 0.12, 0.05]),
        "target_right": ([x + 0.055, y, 0.025], [0.01, 0.12, 0.05]),
        "target_back": ([x, y + 0.055, 0.025], [0.10, 0.01, 0.05]),
        "target_front": ([x, y - 0.055, 0.025], [0.10, 0.01, 0.05]),
    }
    for name, (position, size) in parts.items():
        require_success(
            sim.add_object(
                name=name,
                shape="box",
                position=list(position),
                size=list(size),
                color=blue,
                is_static=True,
            ),
            f"add {name}",
        )
    require_success(
        sim.add_camera(
            name="top",
            position=[0.45, -0.60, 0.42],
            target=[0.06, -0.31, 0.06],
            fov=55,
            width=640,
            height=480,
        ),
        "add top camera",
    )
    require_success(
        sim.add_camera(
            name="wrist",
            position=[0.02, -0.56, 0.18],
            target=[0.04, -0.31, 0.04],
            fov=70,
            width=640,
            height=480,
        ),
        "add wrist camera",
    )
    require_success(
        sim.send_action(dataset_mean_action(), robot_name="so100", n_substeps=600),
        "move SO100 to the dataset-mean pose",
    )
    return sim


def task_diagnostics(sim):
    """Return cube position and the authoritative placed-in-box metric."""
    cube_id = mujoco.mj_name2id(sim.mj_model, mujoco.mjtObj.mjOBJ_BODY, "cube")
    position = np.asarray(sim.mj_data.xpos[cube_id], dtype=float).copy()
    inside_box_xy = bool(
        np.all(np.abs(position[:2] - BOX_CENTER) < np.array([0.045, 0.045]))
    )
    return {
        "cube_position_m": np.round(position, 4).tolist(),
        "placed_in_box": inside_box_xy and position[2] < 0.10,
    }


sim = build_scene()
print("Initial task state:", task_diagnostics(sim))

## 4. Inspect the Native `strands-robots` Tool

`Robot("so100")` is itself a stateful Strands `AgentTool`. The agent sees one JSON tool with an `action` field. Its schema includes scene operations, sensors, recording, and `run_policy`. A general robot agent could receive this full tool directly.

In [ ]:
native_schema = sim.tool_spec["inputSchema"]["json"]
native_actions = native_schema["properties"]["action"]["enum"]

print("Agent tool name:", sim.tool_name)
print("Published actions:", len(native_actions))
print("Contains run_policy:", "run_policy" in native_actions)
print("policy_config schema:", native_schema["properties"]["policy_config"])
print("stop_when schema:", native_schema["properties"]["stop_when"])

## 5. Load the Fine-Tuned Policy

The checkpoint was trained on simulation-native camera names and radians. We construct the embodiment adapter explicitly and load the model once. Every agent-requested segment will reuse this same in-memory policy object.

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
finetuned_embodiment = {
    "name": "so100_smolvla_sim_finetuned",
    "obs_rename": {
        "wrist": "observation.images.wrist",
        "top": "observation.images.top",
    },
    "state_keys": JOINT_KEYS,
    "action_keys": JOINT_KEYS,
    "dim_policy": "strict",
    "state_units": "radians",
    "action_units": "radians",
}

finetuned_policy = create_policy(
    "lerobot_local",
    pretrained_name_or_path=str(LOCAL_MODEL_DIR),
    policy_type="smolvla",
    device=device,
    embodiment=finetuned_embodiment,
    strict_keys=True,
)

print("Policy adapter:", type(finetuned_policy).__name__)
print("Device:", device)
print("Execution horizon:", finetuned_policy.execution_horizon)

## 6. Define the Agent's State and Deterministic Stop Condition

The LLM must not own safety or resource limits. Python state enforces a maximum of 400 control steps. The `inside_region` predicate is evaluated by `strands-robots` after each applied action, so a successful rollout can stop before consuming the complete segment.

In [ ]:
import json
from dataclasses import dataclass, field

SEGMENT_STEPS = 200
MAX_TOTAL_STEPS = 400
VIDEO_DIR = Path("outputs/agentic-pick").resolve()
SUCCESS_STOP_WHEN = {
    "predicate": "inside_region",
    "body": "cube",
    "min": [float(BOX_CENTER[0] - 0.045), float(BOX_CENTER[1] - 0.045), 0.0],
    "max": [float(BOX_CENTER[0] + 0.045), float(BOX_CENTER[1] + 0.045), 0.10],
}


@dataclass
class AgenticLoopState:
    """Track the hard policy budget and every completed segment."""
    segment_steps: int
    max_total_steps: int
    total_steps_used: int = 0
    attempts: list[dict] = field(default_factory=list)
    terminal_error: str | None = None

    @property
    def remaining_steps(self):
        return max(0, self.max_total_steps - self.total_steps_used)


loop_state = AgenticLoopState(SEGMENT_STEPS, MAX_TOTAL_STEPS)
print("Stop condition:", SUCCESS_STOP_WHEN)
print("Total policy budget:", loop_state.max_total_steps)

## 7. Build Two Least-Privilege Agent Tools

Instead of exposing all simulation actions, this experiment gives the agent only:

- `inspect_pick_task()`: read deterministic state without moving the robot;
- `run_smolvla_segment()`: execute one bounded policy segment.

The second tool returns both `run_status` and `task_success`. This prevents the agent from confusing successful software execution with successful manipulation.

In [ ]:
from strands import tool


def result_json(result):
    """Extract the first JSON block from a strands-robots result."""
    return next(
        (
            item["json"]
            for item in result.get("content", [])
            if isinstance(item, dict) and isinstance(item.get("json"), dict)
        ),
        {},
    )


def result_text(result):
    """Join human-readable text blocks from a tool result."""
    return " | ".join(
        str(item["text"])
        for item in result.get("content", [])
        if isinstance(item, dict) and item.get("text")
    )


def tool_envelope(payload, status="success"):
    """Return the standard Strands status/content envelope."""
    return {
        "status": status,
        "content": [
            {"text": json.dumps(payload, sort_keys=True)},
            {"json": payload},
        ],
    }


@tool
def inspect_pick_task() -> dict:
    """Read the authoritative physical task metric without moving the robot."""
    diagnostics = task_diagnostics(sim)
    return tool_envelope({
        "task_success": bool(diagnostics["placed_in_box"]),
        "diagnostics": diagnostics,
        "segments_completed": len(loop_state.attempts),
        "steps_used": loop_state.total_steps_used,
        "remaining_steps": loop_state.remaining_steps,
    })

### Wrap the real policy rollout

This second tool is the action boundary. It enforces the remaining budget, invokes `sim.run_policy()` with the preloaded SmolVLA, records one segment, and returns deterministic task feedback to the agent.

In [ ]:
@tool
def run_smolvla_segment() -> dict:
    """Run the loaded SmolVLA for one bounded segment and report task state."""
    diagnostics = task_diagnostics(sim)
    if diagnostics["placed_in_box"]:
        return tool_envelope({
            "run_status": "skipped",
            "task_success": True,
            "steps_used": 0,
            "remaining_steps": loop_state.remaining_steps,
            "diagnostics": diagnostics,
        })
    if loop_state.terminal_error is not None:
        return tool_envelope({
            "error": "A previous rollout ended with a software error.",
            "detail": loop_state.terminal_error,
            "task_success": False,
            "remaining_steps": loop_state.remaining_steps,
        }, status="error")
    if loop_state.remaining_steps == 0:
        return tool_envelope({
            "error": "The policy-step budget is exhausted.",
            "task_success": False,
            "steps_used": loop_state.total_steps_used,
            "remaining_steps": 0,
        }, status="error")

    steps = min(loop_state.segment_steps, loop_state.remaining_steps)
    segment_number = len(loop_state.attempts) + 1
    VIDEO_DIR.mkdir(parents=True, exist_ok=True)
    video_path = VIDEO_DIR / f"segment_{segment_number:02d}.mp4"

    result = sim.run_policy(
        robot_name="so100",
        policy_object=finetuned_policy,
        instruction=INSTRUCTION,
        n_steps=steps,
        control_frequency=FPS,
        fast_mode=True,
        video={"path": str(video_path), "camera": "top", "fps": FPS},
        stop_when=SUCCESS_STOP_WHEN,
    )
    report = result_json(result)
    measured_steps = int(report.get("steps_used", report.get("n_steps", 0)) or 0)
    loop_state.total_steps_used += min(measured_steps, loop_state.remaining_steps)
    diagnostics = task_diagnostics(sim)

    attempt = {
        "segment": segment_number,
        "run_status": result.get("status"),
        "task_success": bool(diagnostics["placed_in_box"]),
        "steps_requested": steps,
        "steps_used": measured_steps,
        "remaining_steps": loop_state.remaining_steps,
        "stopped_reason": report.get("stopped_reason"),
        "action_errors": report.get("action_errors"),
        "partial_action_failure_rate": report.get("partial_action_failure_rate"),
        "diagnostics": diagnostics,
        "video_path": str(video_path),
        "error_text": result_text(result) if result.get("status") != "success" else None,
    }
    loop_state.attempts.append(attempt)
    if result.get("status") != "success":
        loop_state.terminal_error = attempt["error_text"] or "Unknown run_policy error."
    return tool_envelope(
        attempt,
        status="success" if result.get("status") == "success" else "error",
    )


agent_tools = [inspect_pick_task, run_smolvla_segment]
for agent_tool in agent_tools:
    print(agent_tool.tool_name, agent_tool.tool_spec["inputSchema"])

## 8. Write the Supervisor Contract

The system prompt defines the agent's task-level responsibilities. The hard budget remains enforced by Python even if the model ignores these instructions.

In [ ]:
SYSTEM_PROMPT = f"""You are a high-level robot task supervisor.

The SmolVLA policy, not you, controls the robot joints. You may only:
1. call inspect_pick_task to read the deterministic physical task metric;
2. call run_smolvla_segment to let the policy act for up to {SEGMENT_STEPS} control steps.

Follow this closed loop:
- Inspect before acting.
- If task_success is already true, stop.
- Otherwise run one policy segment and inspect the returned task_success.
- If it is false and remaining_steps is positive, inspect once more and run another segment.
- Stop immediately on task_success=true, a software/tool error, or exhausted budget.

The total policy budget is {MAX_TOTAL_STEPS} steps. Never infer success from
run_status='success': that only means the software loop ran. Report success
only when task_success=true. If the budget ends with task_success=false,
report a physical task failure instead of claiming that reasoning repaired
the low-level policy."""

print(SYSTEM_PROMPT)

## 9. Create and Run the Strands Agent

The Bedrock model reasons over tool results at task cadence. It never receives an API for individual joint commands.

In [ ]:
import boto3
from strands import Agent
from strands.models import BedrockModel

AWS_REGION = os.environ.get("AWS_REGION") or boto3.Session().region_name or "us-east-1"
AGENT_MODEL_ID = os.environ.get(
    "STRANDS_MODEL_ID",
    "global.anthropic.claude-sonnet-4-6",
)

agent_model = BedrockModel(model_id=AGENT_MODEL_ID, region_name=AWS_REGION)
agent = Agent(
    model=agent_model,
    tools=agent_tools,
    system_prompt=SYSTEM_PROMPT,
    callback_handler=None,
)

print("Agent model:", AGENT_MODEL_ID)
print("AWS region:", AWS_REGION)

### Give the agent one task-level goal

Calling the agent starts the observe–decide–act loop. The model may choose when to invoke the two tools, while Python continues to enforce the policy-step limit and authoritative success metric.

In [ ]:
agent_result = agent(
    "Complete the simulated SO100 pick-and-place task. "
    "Use the available tools, obey the step budget, and give a concise final verdict."
)
print(agent_result)

## 10. Audit the Outcome

Agent prose is not the experiment record. We inspect the Python-owned attempt history and recompute the deterministic physical metric.

In [ ]:
import pandas as pd
from IPython.display import Video, display

display(pd.DataFrame(loop_state.attempts))
print("Final deterministic diagnostics:", task_diagnostics(sim))
print("Total policy steps used:", loop_state.total_steps_used)

for attempt in loop_state.attempts:
    video_path = Path(attempt["video_path"])
    if video_path.is_file():
        print(f"Segment {attempt['segment']}: {video_path}")
        display(Video(str(video_path), embed=True, width=640))

## What the Agent Adds — and What It Does Not

The agent adds task-level orchestration: tool selection, bounded continuation, interpretation of deterministic feedback, and a final explanation. It does not improve SmolVLA weights, invent missing grasp skill, or replace the real-time controller.

On a physical robot the same separation applies. SmolVLA may run on the robot or a nearby inference computer, while the Strands Agent can run elsewhere and issue bounded task requests. Production hardware still requires independent safety interlocks, authorization, timeouts, and emergency stop.